# Processing stackSentinel over CA

In [ ]:
import os
#os.environ["OMP_NUM_THREADS"] = "4" # export OMP_NUM_THREADS=4
#os.environ["OPENBLAS_NUM_THREADS"] = "1" # export OPENBLAS_NUM_THREADS=4 
#os.environ["MKL_NUM_THREADS"] = "6" # export MKL_NUM_THREADS=6
#os.environ["VECLIB_MAXIMUM_THREADS"] = "4" # export VECLIB_MAXIMUM_THREADS=4
#os.environ["NUMEXPR_NUM_THREADS"] = "6" # export NUMEXPR_NUM_THREADS=6

In [ ]:
import site
from pathlib import Path
import subprocess
import numpy as np
import time as _time
import zipfile
import sys

import requests
from lxml import etree
import urllib.request
from urllib.parse import urljoin

import rasterio
from rasterio import logging
log = logging.getLogger()
log.setLevel(logging.ERROR)

import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from shapely import Polygon

# Plotting modules
from IPython import display
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import cartopy.io.img_tiles as cimgt

# isce2
import isce
isce_application_path = Path(isce.isce_path) / 'applications'
os.environ['PATH'] += (':' + str(isce_application_path))

# TopsStack aux modules
import asf_search as asf
asf.constants.INTERNAL.CMR_TIMEOUT = 120
import eof 
from dem_stitcher.stitcher import stitch_dem


# --- 1. Leer fechas 
# Pair selection optimization modules
import itertools
from datetime import datetime
import networkx as nx

In [ ]:
# Additional functions
def make_geodataframe(asf_search_object):
    df = pd.DataFrame(asf_search_object.properties, dtype=str, index=[0])
    df['wkt'] = Polygon(asf_search_object.geometry['coordinates'][0]).wkt
    gs = gpd.GeoSeries.from_wkt(df['wkt'])
    gdf = gpd.GeoDataFrame(df, geometry=gs, crs="EPSG:4326")
    return gdf

# Add a WGS84 reference tag to the DEM XML so ISCE treats the DEM as ellipsoidal/geodetic.
def tag_dem_xml_as_ellipsoidal(dem_path: Path) -> str:
    xml_path = str(dem_path) + '.xml'
    if not Path(xml_path).exists():
        raise FileNotFoundError(f"DEM XML file not found: {xml_path}")
    tree = etree.parse(xml_path)
    root = tree.getroot()

    y = etree.Element("property", name='reference')
    etree.SubElement(y, "value").text = "WGS84"
    etree.SubElement(y, "doc").text = "Geodetic datum"

    root.insert(0, y)
    with open(xml_path, 'wb') as file:
        file.write(etree.tostring(root, pretty_print=True))
    return xml_path


# Fix ISCE image XML metadata by invoking ISCE's fixImageXml.py (runs with --full)
# and return the raster path.
def fix_image_xml(isce_raster_path: str) -> str:
    isce_application_path = Path(f'{site.getsitepackages()[0]}'
                                '/isce/applications/')

    fix_cmd = [f'{isce_application_path}/fixImageXml.py',
               '-i',
               str(isce_raster_path),
               '--full']
    fix_cmd_line = ' '.join(fix_cmd)
    subprocess.check_call(fix_cmd_line, shell=True)
    print(fix_cmd_line)
    return isce_raster_path

# DEM = Digital Elevation Model
def download_dem_for_isce2(extent: list,
                           dem_name: str = 'glo_30',
                           dem_dir: Path = None,
                           dem_res = 0.0002777777777777777775,
                           buffer: float = .004) -> dict:
    """
    Parameters
    ----------
    extent : list
        [xmin, ymin, xmax, ymin] for epsg:4326 (i.e. (x, y) = (lon, lat))
    dem_name : str, optional
        See names in `dem_stitcher`
    dem_dir: Path, optional
        
    buffer : float, optional
        In degrees, by default .001, which is .5 km at equator
    Returns
    -------
    Path
    """
    dem_dir = dem_dir or Path(f'{dem_name}')
    dem_dir.mkdir(exist_ok=True, parents=True)

    extent_geo = box(*extent)
    extent_buffered = list(extent_geo.buffer(buffer).bounds)
    extent_buffered = [np.floor(extent_buffered[0]), np.floor(extent_buffered[1]),
                       np.ceil(extent_buffered[2]), np.ceil(extent_buffered[3])]

    # you can remove this parameter if you don't mind resolution of original DEM
    dem_array, dem_profile = stitch_dem(extent_buffered,
                                        dem_name,
                                        dst_ellipsoidal_height=True,
                                        dst_area_or_point='Point',
                                        # ensures square resolution
                                        dst_resolution=dem_res
                                        )

    dem_path = dem_dir / 'full_res.dem.wgs84'
    print(dem_path)
    dem_array[np.isnan(dem_array)] = 0.

    dem_profile_isce = dem_profile.copy()
    dem_profile_isce['nodata'] = None
    dem_profile_isce['driver'] = 'ISCE'
    # remove keys that do not work with ISCE gdal format
    [dem_profile_isce.pop(key) for key in ['blockxsize', 'blockysize', 'compress', 'interleave', 'tiled']]

    with rasterio.open(dem_path, 'w', **dem_profile_isce) as ds:
        ds.write(dem_array, 1)
        
    dem_xml = tag_dem_xml_as_ellipsoidal(dem_path)
    fix_image_xml(dem_xml)

    return dem_xml

# Adjustable parameters

In [ ]:
# Work Directory and Area of Interest
NOTEBOOK_DIR = Path.cwd()   # capture this ONCE, at the very top of the notebook, before any os.chdir() calls
work_dir = NOTEBOOK_DIR / 'run_workdir'
aoi = [37, 39, -120.5, -118] # snwe 

# Download parameters
track = 144 # Sentinel-1 track number
start_date = '2023-02-01'
end_date = '2023-02-16'
flight_direction = 'DSC' # 'ASC' or 'DSC', ASC = ascending, DSC = descending
n_processes = 2 # number of threads for downloading SLCs
dem_name = 'glo_30'


# Coherence Proxy Calibration — Mexico City
# Ref: Smittarello et al. (2022), JGR Solid Earth

# Seasonal contribution (w1)
# Mexico City: rainy season May–Oct → DOYlow ≈ 230 (mid-Aug peak decorrelation)
DOYlow = 230   # day of year with lowest coherence
alpha  = 2     # width of low-coherence period [1–5]; wider = longer season

# Temporal decorrelation (w2)
beta   = 0.015  # decay rate [day⁻¹]; urban MX ≈ moderate, less than tropical

# Spatial decorrelation (w3)
gamma  = 0.02   # decay rate [m⁻¹]; Sentinel-1 small tube → nearly irrelevant

# Expected coherence range on ROI
Mxc = 0.70   # max expected mean coherence
Mnc = 0.25   # min expected mean coherence

# Graph optimization criterion
k_opt = 2    # max connections per image as primary AND as secondary
             # total pairs per image = 2k; k=3 → paper's recommended sweet spot

# Baseline thresholds for candidate pool
max_btemp_days = 40   # temporal [days]  — keep large, optimizer will prune
max_bperp_m    =  20   # perpendicular [m] — Sentinel-1 tight tube

In [ ]:
## Set-up processing directory structure

# Sentinel-1 SLCs folder, SLC = Single Look Complex
slc_dir = work_dir / flight_direction / str(track) / 'data'
slc_dir.mkdir(exist_ok=True, parents=True)

# GLO-30 DEM folder
dem_dir = work_dir / 'DEM'
dem_dir.mkdir(exist_ok=True, parents=True)
dem_path = dem_dir / 'full_res.dem.wgs84'

# Sentinel-1 ORBIT data folder
orbit_dir = work_dir / 'ORBIT'
orbit_dir.mkdir(exist_ok=True, parents=True)

# Sentinel-1 AUX CAL-file folder
aux_dir = work_dir / 'AUX'
aux_dir.mkdir(exist_ok=True, parents=True)

# stackSentinel dir
isce_run_dir = slc_dir.parent / 'isce'
isce_run_dir.mkdir(exist_ok=True, parents=True)

run_dir = isce_run_dir / 'run_files'
run_ifg_dir = isce_run_dir / 'run_ifg_files'

## Download Sentinel-1 images

### Crear una cuenta en NASA Earthdata y obtener credenciales
 
Las imágenes Sentinel-1 se descargan desde **Alaska Satellite Facility (ASF)**,
que usa el sistema de autenticación de NASA Earthdata.
 
#### Registrar cuenta
 
1. [https://urs.earthdata.nasa.gov/users/new](https://urs.earthdata.nasa.gov/users/new)
2. Completar el formulario de registro.
 
Una vez con sesión iniciada en Earthdata:
 
3. Ir a **Applications → Authorized Apps**
2. Buscar y aprobar las siguientes aplicaciones:
   - **Alaska Satellite Facility Data Access**
   - **ASF Data Access**
 
### Guardar las credenciales en `~/.netrc`
 
`asf_search` y otras herramientas del ecosistema SAR usan el archivo `~/.netrc`
estándar de Unix para autenticarse automáticamente sin exponer contraseñas en el
código.
 
#### Crear o editar el archivo
 
Agrega el siguiente bloque al final del archivo, reemplazando los valores:
 
```
machine urs.earthdata.nasa.gov
    login TU_USUARIO_EARTHDATA
    password TU_CONTRASEÑA_EARTHDATA
```
 
#### Aplicar permisos restrictivos
 
```bash
chmod 600 ~/.netrc
```
 
> **Sin este paso**, herramientas como `curl` y `asf_search` ignorarán el archivo o fallarán silenciosamente.


In [ ]:
# Create aoi_polygon
aoi_polygon = Polygon([(aoi[2], aoi[0]), (aoi[3], aoi[0]),
                       (aoi[3], aoi[1]), (aoi[2], aoi[1])])
                       
# Increase the aoi for 0.1 deg by adding buffer
aoi_polygon = aoi_polygon.buffer(0.1) 

# Prepare flight_directionf for asf_search
if flight_direction.casefold().startswith('asc'):
    orbit = asf.FLIGHT_DIRECTION.ASCENDING
elif flight_direction.casefold().startswith('dsc'):
    orbit = asf.FLIGHT_DIRECTION.DESCENDING
else:
    raise ValueError(f'Selected flight direction \033[1m"{flight_direction}"\033[0m'
                     f' does not exist! \n '   
                     f'            Available options: "Ascending" or "Descending"')

In [ ]:
# Find number of available Sentinel-1 images
search_results = asf.geo_search(
    platform=asf.PLATFORM.SENTINEL1,
    intersectsWith=aoi_polygon.wkt,
    start= start_date,
    end=end_date,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    beamMode=asf.BEAMMODE.IW,
    relativeOrbit=track, # Change the path
    flightDirection=orbit,
)
print(f'Available {len(search_results)} Sentinel-1 SLCs to download!')


# Loop to all search results
gdfs = [make_geodataframe(result) for result in search_results]

# Concatenate all GeoDataFrames into one
gdfs = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True),
                        geometry='geometry', crs="EPSG:4326")

# Get the preview of Dataframe
gdfs.head()

In [ ]:
# Plot the results
fig, ax = plt.subplots(1, figsize=[12,8], subplot_kw=dict(projection=ccrs.PlateCarree()))
image = cimgt.GoogleTiles(url = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}')
ax.add_image(image, 6, cmap='gray', zorder=0)
gdfs.exterior.plot(ax=ax)
ax.plot(aoi_polygon.exterior.xy[0], aoi_polygon.exterior.xy[1], 'r-', linewidth=3)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True)
gl.top_labels = False
gl.right_labels = False
gl.xlines = False
gl.ylines = False


In [ ]:
# Download the SLCs one product at a time, with retries.
# Fix: search_results.download() uses a multiprocessing pool that aborts the
# ENTIRE batch if a single product's connection drops mid-download
# (ChunkedEncodingError / IncompleteRead). This loop isolates failures per
# product, retries transient network errors, and reports a final summary
# instead of losing all progress on one bad connection.

MAX_RETRIES = 3
RETRY_DELAY_SEC = 10

failed_downloads = []

for i, product in enumerate(search_results):
    file_name = product.properties['fileName']
    file_path = slc_dir / file_name

    if file_path.exists():
        print(f'[{i+1}/{len(search_results)}] SKIP {file_name} already exists')
        continue

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            product.download(path=str(slc_dir))
            success = True
            print(f'[{i+1}/{len(search_results)}] Downloaded {file_name}')
            break
        except Exception as e:
            print(f'[{i+1}/{len(search_results)}] Attempt {attempt}/{MAX_RETRIES} '
                  f'failed for {file_name}: {e}')
            # clean up partial file so we don't leave a truncated zip behind
            if file_path.exists():
                file_path.unlink()
            if attempt < MAX_RETRIES:
                _time.sleep(RETRY_DELAY_SEC)

    if not success:
        failed_downloads.append(file_name)

print(f'\n{len(search_results) - len(failed_downloads)}/{len(search_results)} SLCs downloaded successfully.')
if failed_downloads:
    print(f'{len(failed_downloads)} failed after {MAX_RETRIES} attempts:')
    for f in failed_downloads:
        print(f'   - {f}')
else:
    print('All SLCs downloaded correctly.')

## Download DEM
https://github.com/ACCESS-Cloud-Based-InSAR/dem-stitcher

In [ ]:
download_dem_for_isce2(extent=[aoi[2], aoi[0], aoi[3], aoi[1]],
                       dem_name=dem_name,
                       dem_dir=dem_dir,
                       buffer=0.5)

## Download ORBIT
https://github.com/scottstanie/sentineleof/blob/master/eof/download.py

In [ ]:
# eof = Earth Observation Finder, a module to download auxiliary files for Sentinel-1 SLCs
eof.download.main(search_path=slc_dir, save_dir=orbit_dir)

## AUX DATA
https://aux.sentinel1.eo.esa.int/

In [ ]:
api_url = 'https://sar-mpc.eu/api/v1/?product_type=AUX_CAL&adf__active=1'
response = requests.get(api_url)
response.raise_for_status()
api_data = response.json()

results = api_data.get('results', [])
print(f'Found {len(results)} AUX_CAL products from the API')

failed_products = []

for product in results:
    file_name = product['physical_name']
    file_url = product['remote_url']
    file_path = aux_dir / file_name

    # Fix: quitar TODAS las extensiones conocidas (.zip y .SAFE), no solo una
    # (.stem por si solo quita una extension y no detecta la carpeta ya extraida)
    base_name = file_name
    for ext in ('.zip', '.SAFE'):
        if base_name.endswith(ext):
            base_name = base_name[: -len(ext)]
    extracted_dir = aux_dir / base_name

    if file_path.exists() or extracted_dir.exists():
        print(f'SKIP {file_name} exists')
        continue

    try:
        response = requests.get(file_url, timeout=60)
        response.raise_for_status()
        file_path.write_bytes(response.content)

        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(aux_dir)
        file_path.unlink()
        print(f'Downloaded: {file_name}')

    except requests.RequestException as e:
        print(f'Failed to download: {file_name} — {e}')
        failed_products.append(file_name)
        continue

    except zipfile.BadZipFile as e:
        print(f'Failed to extract: {file_name} — {e}')
        failed_products.append(file_name)
        if file_path.exists():
            file_path.unlink()  # limpia el zip corrupto para no confundir el proximo skip-check
        continue

# Resumen final
if failed_products:
    print(f'{len(failed_products)} failed:')
    for f in failed_products:
        print(f'   - {f}')
else:
    print('All AUX_CAL products downloaded correctly.')

# Stack Sentinel - make coregistrated SLC stack

In [ ]:
# # We need to add isce2/contrib/stack/ directory to our PATH env variable
# isce2_stack_dir = Path('/home/erickleon/tools/isce22/src/isce2/contrib/stack')
# os.environ['PATH'] += (':' + str(isce2_stack_dir / 'topsStack'))
# # os.environ['PYTHONPATH'] = str(isce2_stack_dir) 
# os.environ['PYTHONPATH'] = str(isce2_stack_dir) + ':' + os.environ.get('PYTHONPATH', '')

# # Go to work_dir processing directory
# os.chdir(isce_run_dir)
# print(f'Work directory; {os.getcwd()}')

# Stack Sentinel - make coregistrated SLC stack

# Detecta automáticamente la ruta según el entorno conda activo
conda_prefix = Path(sys.prefix)
isce2_stack_dir = conda_prefix / 'share/isce2'

# Detecta la version de Python del env activo dinamicamente
py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
site_packages = conda_prefix / f'lib/python{py_version}/site-packages'

os.environ['PATH'] += ':' + str(isce2_stack_dir / 'topsStack')

# Fix: anexar site_packages (para que el subprocess encuentre 'isce'), no sobrescribir.
# Ademas, guarda el PYTHONPATH original una sola vez para evitar que se duplique
# si esta celda se re-ejecuta varias veces en el mismo kernel.
_base_pythonpath = os.environ.get('_ORIG_PYTHONPATH')
if _base_pythonpath is None:
    _base_pythonpath = os.environ.get('PYTHONPATH', '')
    os.environ['_ORIG_PYTHONPATH'] = _base_pythonpath

os.environ['PYTHONPATH'] = f"{isce2_stack_dir}:{site_packages}:{_base_pythonpath}"

# Verifica ambas rutas
result = subprocess.run('which stackSentinel.py', shell=True, capture_output=True, text=True)
print(f"stackSentinel.py: {result.stdout.strip()}")
print(f"PYTHONPATH: {os.environ['PYTHONPATH']}")

os.chdir(isce_run_dir)
print(f'Work directory: {os.getcwd()}')

In [ ]:
# Generate config and run file for generation of coregistrated SLCs

processing_bound = ' '.join(str(x) for x in aoi) # SNWE
args = f'stackSentinel.py -s {slc_dir} -d {dem_path} -b "{processing_bound}" -a {aux_dir} -o {orbit_dir} -C NESD  -W slc'
args += ' --num_proc4topo 10 --num_proc 10' # to add multi threading
print(args)

# Creates configs and run_files
# Fix: valida returncode + captura stdout/stderr en vez de fallar silenciosamente
# (un COMPLETED/returncode==0 en apariencia no garantiza que run_files se generaran)
result = subprocess.run(
    args, shell=True, close_fds=True,
    capture_output=True, text=True
)

if result.returncode != 0:
    print('STDOUT:\n', result.stdout)
    print('STDERR:\n', result.stderr)
    raise RuntimeError(f'stackSentinel.py fallo con codigo {result.returncode}')

print(f'stackSentinel.py completado (codigo {result.returncode})')

# List created run files
run_files = list(run_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
print(f"isce2_stack_dir existe: {isce2_stack_dir.exists()}")
print(f"topsStack existe: {(isce2_stack_dir / 'topsStack').exists()}")
print(f"stackSentinel.py existe: {(isce2_stack_dir / 'topsStack/stackSentinel.py').exists()}")
#  ¿Existe run_dir y qué contiene?
print(f"run_dir: {run_dir}")
print(f"Existe: {run_dir.exists()}")
print(f"Archivos: {list(run_dir.glob('*'))}")

In [ ]:
# Step 1 - unpack_topo_reference
# Directory: reference and geom_reference
run_file = list(run_dir.glob('run_01*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 2
# Directory: secondarys 
run_file = list(run_dir.glob('run_02*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 3 average baseline
# Directory: baselines
run_file = list(run_dir.glob('run_03*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 4 extract_burst_overlaps
# Directory: reference/overlap
run_file = list(run_dir.glob('run_04*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)

out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 5 overlap_geo2rdr
# Directory: coreg_secondarys
run_file = list(run_dir.glob('run_05*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 6 overlap_resample
# Directory: coreg_secondarys
run_file = list(run_dir.glob('run_06*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 7 pairs_misreg
# Directory: ESD 
run_file = list(run_dir.glob('run_07*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 8 pairs_misreg
# Directory: timeseries_misreg
run_file = list(run_dir.glob('run_08*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 9 pairs_misreg
# Directory: fullBurst_geo2rdr!
run_file = list(run_dir.glob('run_09*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 10 pairs_misreg
# Directory: fullBurst_resample!
run_file = list(run_dir.glob('run_10*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 11 pairs_misreg
# Directory: extract_stack_valid_region'
run_file = list(run_dir.glob('run_11*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 12 pairs_misreg
# Directory: merge_reference_secondary_slc
run_file = list(run_dir.glob('run_12*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 13 pairs_misreg
# Directory: grid_baseline
run_file = list(run_dir.glob('run_13*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
baseline_dir = isce_run_dir / 'baselines'

# ── 1. DIAGNOSE first ───────────────────────────────────────────────────────
print(f"baseline_dir exists: {baseline_dir.exists()}")
all_files = list(baseline_dir.glob('**/*'))
print(f"Total items under baseline_dir: {len(all_files)}")
for f in all_files[:10]:                     # preview first 10
    print(f"  {'DIR ' if f.is_dir() else 'FILE'} {f.relative_to(baseline_dir)}")
    if f.is_file():
        try:
            print(f"       content preview: {f.read_text().strip()[:120]!r}")
        except Exception as e:
            print(f"       read error: {e}")

In [ ]:
records = []
baseline_dir = isce_run_dir / 'baselines'

for pair_dir in sorted(baseline_dir.iterdir()):
    if not pair_dir.is_dir():
        continue
    parts = pair_dir.name.split('_')
    if len(parts) != 2:
        continue
    ref, sec = parts

    for f in pair_dir.glob('*.txt'):
        try:
            text = f.read_text().strip()
            bp = None
            for line in text.splitlines():
                if 'bperp' in line.lower():    # matches "Bperp (average):"
                    bp = float(line.split()[-1])
                    break
            if bp is not None:
                records.append({'ref': ref, 'sec': sec, 'bperp': bp})
                break
        except Exception as e:
            print(f"  Skipped {f}: {e}")

if not records:
    raise RuntimeError(
        f"No baseline records parsed from {baseline_dir}. "
        "Check that stackSentinel step 3 (average_baseline) completed successfully."
    )

pairs_df = pd.DataFrame(records)

def to_dt(date_str):
    return datetime.strptime(date_str, '%Y%m%d')

def to_doy(date_str):
    return to_dt(date_str).timetuple().tm_yday

pairs_df['date_ref'] = pairs_df['ref'].apply(to_dt)
pairs_df['date_sec'] = pairs_df['sec'].apply(to_dt)
pairs_df['btemp']    = (pairs_df['date_sec'] - pairs_df['date_ref']).dt.days
pairs_df['doy_ref']  = pairs_df['ref'].apply(to_doy)
pairs_df['doy_sec']  = pairs_df['sec'].apply(to_doy)

print(f"Total candidate pairs (within baselines): {len(pairs_df)}")
pairs_df.head()

In [ ]:
# --- 2. Calcular coherence proxy para cada par ---

def coherence_proxy(doy_p, doy_s, bt, bp,
                    DOYlow, alpha, beta, gamma, Mxc, Mnc,
                    a=0.30, b=0.43, c=0.02):
    """
    Proxy de coherencia basado en Smittarello et al. (2022) Eq. 1-4.
    Coeficientes a,b,c se invierten con calibración; usar defaults
    del paper (DLM) como punto de partida y refinar con datos MX.
    """

    # w1: contribución estacional
    w1 = np.abs(
        np.sin((doy_p + (365 - DOYlow)) / 365 * np.pi) *
        np.sin((doy_s + (365 - DOYlow)) / 365 * np.pi)
    ) ** alpha

    # w2: decorrelación temporal
    w2 = (Mxc - Mnc) * np.exp(-beta * np.abs(bt)) + Mnc

    # w3: decorrelación espacial
    w3 = (Mxc - Mnc) * np.exp(-gamma * np.abs(bp)) + Mnc

    # Proxy combinado (sin normalizar — suficiente para ranking)
    return a * w1 + b * w2 + c * w3


pairs_df['proxy_w'] = pairs_df.apply(
    lambda r: coherence_proxy(
        r['doy_ref'], r['doy_sec'], r['btemp'], r['bperp'],
        DOYlow=DOYlow, alpha=alpha, beta=beta, gamma=gamma,
        Mxc=Mxc, Mnc=Mnc
    ), axis=1
)

# Filtrar por baseline criteria
pairs_filtered = pairs_df[
    (pairs_df['btemp'].abs() <= max_btemp_days) &
    (pairs_df['bperp'].abs() <= max_bperp_m)
].copy()

print(f"Pairs after baseline filtering: {len(pairs_filtered)}")
print(pairs_filtered[['ref','sec','btemp','bperp','proxy_w']].sort_values('proxy_w', ascending=False).head(10))

In [ ]:
# --- 3. Optimización del grafo (Algorithm de Smittarello et al. §3.2.2) ---

def optimize_pair_graph(pairs_df, k=3):
    """
    Implementa el algoritmo de optimización de Smittarello et al. (2022).
    
    Para cada nodo (fecha):
    - out-degree (como Primary) limitado a k
    - in-degree (como Secondary) limitado a k
    Si hay exceso, se eliminan los arcos con menor proxy_w.
    
    Returns: DataFrame con los pares seleccionados.
    """
    # Grafo dirigido ponderado: ref → sec, peso = proxy_w
    G = nx.DiGraph()
    for _, row in pairs_df.sort_values('proxy_w', ascending=False).iterrows():
        G.add_edge(row['ref'], row['sec'], weight=row['proxy_w'],
                   btemp=row['btemp'], bperp=row['bperp'])

    # Ordenar nodos cronológicamente
    nodes = sorted(G.nodes())

    selected_edges = set()

    for node in nodes:
        out_edges = list(G.out_edges(node, data=True))
        
        if len(out_edges) <= k:
            # Mantener todos
            for u, v, d in out_edges:
                selected_edges.add((u, v))
        else:
            # Dividir en dos clases según in-degree del nodo destino
            keep_forced = []   # v con in-degree <= k (no saturados): forzar retención
            keep_optional = [] # v con in-degree > k: ordenar por peso

            for u, v, d in out_edges:
                in_deg_v = G.in_degree(v)
                if in_deg_v <= k:
                    keep_forced.append((u, v, d['weight']))
                else:
                    keep_optional.append((u, v, d['weight']))

            # Mantener forzados primero
            for u, v, w in keep_forced:
                selected_edges.add((u, v))

            # Completar hasta k con los de mayor peso
            remaining = k - len(keep_forced)
            if remaining > 0:
                keep_optional.sort(key=lambda x: x[2], reverse=True)
                for u, v, w in keep_optional[:remaining]:
                    selected_edges.add((u, v))

    # Filtrar DataFrame
    mask = pairs_df.apply(
        lambda r: (r['ref'], r['sec']) in selected_edges, axis=1
    )
    optimized = pairs_df[mask].copy()
    return optimized


optimized_pairs = optimize_pair_graph(pairs_filtered, k=k_opt)

print(f"\nPares originales (dentro de baseline criteria): {len(pairs_filtered)}")
print(f"Pares optimizados (k={k_opt}): {len(optimized_pairs)}")
print(f"Reducción: {100*(1 - len(optimized_pairs)/len(pairs_filtered)):.1f}%")

In [ ]:
# --- 4. Visualizar la red de pares optimizada ---

fig, axes = plt.subplots(1, 2, figsize=[16, 5])

for ax, df, title in zip(axes,
                          [pairs_filtered, optimized_pairs],
                          ['Original (baseline criteria)', f'Optimized (k={k_opt})']):
    for _, row in df.iterrows():
        ax.plot([row['date_ref'], row['date_sec']],
                [row['bperp'], row['bperp']], 'b-', alpha=0.3, lw=0.8)
        ax.plot([row['date_ref'], row['date_sec']],
                [row['bperp'], row['bperp']], 'ko', markersize=2)
    ax.set_title(f"{title}\n{len(df)} pairs")
    ax.set_xlabel('Date'); ax.set_ylabel('Bperp (m)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(isce_run_dir / 'optimized_network.png'), dpi=150)
plt.show()

# Generate interferograms


In [ ]:
download_file = isce2_stack_dir / 'topsStack/interferogramStack.py'

# Fix: parametrizado via variable de entorno en vez de hardcodear la ruta local
# de una sola maquina (evita el mismo problema de paths heredado del entorno de 'doki').
# En el cluster: exporta IFG_STACK_SRC=/mnt/beegfs/insarlab/erick/isce2_topsStack_ifg_network/interferogramStack.py
source_file = Path(os.environ.get(
    'IFG_STACK_SRC',
    Path.home() / 'isce2_topsStack_ifg_network' / 'interferogramStack.py'
))

download_file.parent.mkdir(parents=True, exist_ok=True)
download_file.write_text(source_file.read_text())

download_file.chmod(0o755)
subprocess.run('interferogramStack.py -h', shell=True)

# Guardar lista de pares optimizados
pair_list_path = isce_run_dir / 'optimized_pair_list.txt'
with open(pair_list_path, 'w') as f:
    for _, row in optimized_pairs.iterrows():
        f.write(f"{row['ref']} {row['sec']}\n")

print(f"Saved {len(optimized_pairs)} pairs to {pair_list_path}")

# El script ahora consume pair_list directamente, así que no hace falta filtrar run_files manualmente


In [ ]:
ifg_args = dict(
            network = 'sequential',       # Se mantiene por compatibilidad; pair_list manda si se define
            pair_list = str(pair_list_path),
            num_connections = 1,          # connection number of interferograms between each date for sequential network
            periodic_connections = None,  # number of periodic interferograms in days [180, 365], str or list
            periodic_tolerance = 1,       # tolerancsequentiale for selection of periodic interferograms around the defined period
            single_reference_date = None, # reference date for single reference network, e.g. 2015-01-23
            start_date = None,            # Start date for interferogram network generation, e.g. 2015-01-23
            end_date = None,              # End date for interferogram network generation, e.g. 2015-01-23
            max_bperp = None,             # Threshold for Maximum Perpendicular baseline [in meters]
            max_btemp = None,             # Threshold for Maximum Temporal baseline [in days]
            azimuth_looks = 3,            # Number of looks in azimuth for interferogram multi-looking
            range_looks = 9,              # Number of looks in range for interferogram multi-looking
            filter_strength = 0.5,        # Filter strength for interferogram filtering
            unw_method = 'snaphu',        # Unwrapping method
            force = True                  # Overwrite run files directory
            )

ifg_cmd_params = dict(
            network = '-n',       
            pair_list = '--pair_list',
            num_connections = '-c',         
            periodic_connections = '-p',  
            periodic_tolerance = '-pt',       
            single_reference_date = '-sr', 
            start_date = '--start_date',           
            end_date = '--end_date',              
            max_bperp = '--max_bperp',             
            max_btemp = '--max_btemp',            
            azimuth_looks = '-z',           
            range_looks = '-r',             
            filter_strength = '-f',        
            unw_method = '-u',        
            force = '--force'                  
            )

ifg_args

In [ ]:
cmd = f'interferogramStack.py -s {isce_run_dir}'
for arg in ifg_args.keys():
    if ifg_args[arg]:
        if arg == 'force':
            cmd +=  ' ' + ifg_cmd_params[arg]
        else:
            cmd +=  ' ' + ifg_cmd_params[arg] + ' ' + str(ifg_args[arg])

print(cmd)
subprocess.run(cmd, shell=True)
display.Image(f'{isce_run_dir}/interferogram_network.png',width=1000, height=1000)

# List the run files
run_files = list(run_ifg_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
# Step 21 
# Directory: generate_burst_igram!
run_file = list(run_ifg_dir.glob('run_21*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 22
# Directory: _merge_burst_igram!
run_file = list(run_ifg_dir.glob('run_22*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 23
# Directory: filter_coherence!
run_file = list(run_ifg_dir.glob('run_23*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 24
# Directory: coarse_interferograms
run_file = list(run_ifg_dir.glob('run_24*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

# Process more interferograms

In [ ]:
# Lets change network to delaunay
ifg_args['network'] = 'delaunay'

In [ ]:
cmd = f'interferogramStack.py -s {isce_run_dir}'
for arg in ifg_args.keys():
    if ifg_args[arg]:
        if arg == 'force':
            cmd +=  ' ' + ifg_cmd_params[arg]
        else:
            cmd +=  ' ' + ifg_cmd_params[arg] + ' ' + str(ifg_args[arg])

print(cmd)
subprocess.run(cmd, shell=True)

# List the run files
run_files = list(run_ifg_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
display.Image(f'{isce_run_dir}/interferogram_network.png',width=500, height=500)